# Tight End Transfer Analysis
Match each tight end’s transfer season to their previous season so their team and statistics can be compared.

In [12]:
import pandas as pd

url = "https://raw.githubusercontent.com/6ikh/NCAATransfers/main/data/transfers_2021_2025_adj.xlsx"
df = pd.read_excel(url)


In [13]:
# Keep tight ends only
te_df = df[df["position"] == "TE"]

In [14]:
# Keep necessary columns only
te_analysis_df = te_df[["season","id","name","position","team" 
                        ,"fbs_conference","power_conf","transferred"
                        ,"transfer_in","prev_season","prev_team"
                        ,"prev_conference","prev_power_conf"
                        ,"transfer_direction","seasons_in_data"
                        ,"teams_in_career","receiving_REC"
                        ,"receiving_YDS","receiving_YPR","receiving_TD"
                        ,"adjPPA"]]

In [15]:
# Keep only transfer-in seasons 
# Exclude previous conference is non-FBS 
te_transfer_seasons = te_analysis_df.loc[
    (te_analysis_df["transfer_in"] == "Y")
    & (te_analysis_df["prev_conference"] != "non-FBS")
].copy()

In [16]:
# Prepare previous season data
previous_seasons = (
    te_analysis_df[
        ["id","season","team",
         "receiving_REC","receiving_YDS","receiving_YPR",
         "receiving_TD","adjPPA"]
    ]
    .rename(columns={
        "season": "prev_season",
        "team": "pre_team",
    })
)

In [17]:
# Match each transfer season to the same player’s previous season using ID and Prev_Season
te_transfer_pairs = te_transfer_seasons.merge(
    previous_seasons,
    on=["id","prev_season"],
    how="left",
    suffixes=("_post","_pre"),
    indicator=True
)

In [18]:
metrics = [
    "receiving_REC",
    "receiving_YDS",
    "receiving_YPR",
    "receiving_TD",
    "adjPPA",
]

for metric in metrics:
    te_transfer_pairs[f"{metric}_change"] = (
        te_transfer_pairs[f"{metric}_post"]
        - te_transfer_pairs[f"{metric}_pre"]
    )
    

In [19]:
te_transfer_pairs

,season,id,name,position,team,fbs_conference,power_conf,transferred,transfer_in,prev_season,...,receiving_YDS_pre,receiving_YPR_pre,receiving_TD_pre,adjPPA_pre,_merge,receiving_REC_change,receiving_YDS_change,receiving_YPR_change,receiving_TD_change,adjPPA_change
0,2023,4576297,A.J. Barner,TE,Michigan,Big Ten,Y,Y,Y,2022.0,...,193.0,7.1,3.0,0.366609,both,-5.0,56.0,4.2,-2.0,0.411583
1,2025,4879044,Ademola Faleye,TE,Washington State,Pac-12,N,Y,Y,2024.0,...,18.0,9.0,0.0,0.649246,both,2.0,5.0,-3.2,1.0,-0.062670
2,2025,4608936,Ahmon Green,TE,Georgia Southern,Sun Belt,N,Y,Y,2023.0,...,147.0,7.7,4.0,0.700734,both,-3.0,19.0,2.7,-4.0,-0.296522
3,2025,4832029,Alex Bauman,TE,Miami,ACC,Y,Y,Y,2024.0,...,212.0,10.6,7.0,0.884823,both,-2.0,-46.0,-1.4,-6.0,-0.493101
4,2025,4433844,Alex Lines,TE,Northwestern,Big Ten,Y,Y,Y,2022.0,...,8.0,8.0,0.0,0.686226,both,3.0,6.0,-4.5,0.0,-0.136339
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
216,2024,4880236,Will Kacmarek,TE,Ohio State,Big Ten,Y,Y,Y,2023.0,...,243.0,11.0,2.0,0.685682,both,-14.0,-157.0,-0.2,-2.0,0.001493
217,2025,5114312,Will Monney,TE,Oklahoma State,Big 12,Y,Y,Y,2024.0,...,138.0,12.5,1.0,0.739030,both,-8.0,-99.0,0.5,-1.0,-0.088533
218,2025,4869390,Wyatt Sullivan,TE,South Florida,American Athletic,N,Y,Y,2024.0,...,155.0,11.9,2.0,0.766778,both,5.0,21.0,-2.1,0.0,-0.167156
219,2024,4877824,Zach Horton,TE,Indiana,Big Ten,Y,Y,Y,2023.0,...,275.0,10.2,6.0,0.771271,both,-6.0,-86.0,-1.2,-2.0,-0.041721


In [25]:
te_np_to_p = te_transfer_pairs[
    te_transfer_pairs["transfer_direction"] == "NP->P"
].copy()

### Summarize Results

In [26]:
te_changes = [
    "receiving_REC_change",
    "receiving_YDS_change",
    "receiving_YPR_change",
    "receiving_TD_change",
    "adjPPA_change",
]

te_np_to_p[te_changes].mean()


receiving_REC_change    -7.795918
receiving_YDS_change   -86.489796
receiving_YPR_change    -0.836735
receiving_TD_change     -0.979592
adjPPA_change           -0.052059
dtype: float64

In [27]:
te_np_to_p[te_changes].median()

receiving_REC_change    -9.000000
receiving_YDS_change   -86.000000
receiving_YPR_change    -1.200000
receiving_TD_change     -1.000000
adjPPA_change           -0.038781
dtype: float64

In [29]:
(te_np_to_p[te_changes] > 0).mean() * 100

receiving_REC_change    17.647059
receiving_YDS_change    15.686275
receiving_YPR_change    43.137255
receiving_TD_change     23.529412
adjPPA_change           39.215686
dtype: float64

### Export Data for Visualization

The transfer analysis is saved for the separate visualization notebook. Run the analysis through the export cell before opening that notebook.

In [46]:
from pathlib import Path

te_data_dir = Path("code/Positions/TE")
if not te_data_dir.exists():
    te_data_dir = Path(".")

te_np_to_p.to_csv(te_data_dir / "te_np_to_p.csv", index=False)
print(f"Saved {len(te_np_to_p)} NP-to-P transfer rows to {te_data_dir / 'te_np_to_p.csv'}")

Saved 51 NP-to-P transfer rows to te_np_to_p.csv
